# LangChain DeepAgents — Practical Examples

This notebook walks through six self-contained examples that cover the core capabilities of [DeepAgents](https://github.com/langchain-ai/deepagents):

1. **Basic agent** with a custom tool
2. **Planning** — automatic task decomposition with `write_todos`
3. **Filesystem** — read, write, and edit files inside the agent
4. **Subagent delegation** — orchestrator + specialist child agent
5. **Streaming output** — live token-by-token responses
6. **Multi-step research pipeline** — the full deep-agent pattern

All examples use the same Azure OpenAI setup as the rest of this repo.

## Setup

In [ ]:
# Install DeepAgents (run once)
# !pip install deepagents langchain-openai python-dotenv azure-identity

In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

set_llm_cache(SQLiteCache(database_path="cache/langchain.db"))

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

In [ ]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    deployment_name="gpt-4o",
    azure_ad_token_provider=token_provider,
    temperature=0,
)

---
## Example 1 — Basic Agent with a Custom Tool

The minimal pattern: pass your LLM instance, a list of tools, and a system prompt.  
DeepAgents adds planning, filesystem, context management, and subagent delegation automatically.

In [ ]:
from langchain_core.tools import tool
from deepagents import create_deep_agent

# --- Define a simple custom tool ---
@tool
def get_stock_price(ticker: str) -> str:
    """Return the latest stock price for a ticker symbol."""
    # Simulated prices — replace with a real API call (e.g. yfinance)
    prices = {"AAPL": 213.49, "MSFT": 452.10, "GOOGL": 189.30, "NVDA": 137.70}
    price = prices.get(ticker.upper())
    if price is None:
        return f"Ticker '{ticker}' not found."
    return f"{ticker.upper()}: ${price:.2f}"


@tool
def calculate_percentage_change(old_price: float, new_price: float) -> str:
    """Calculate the percentage change between two prices."""
    change = ((new_price - old_price) / old_price) * 100
    direction = "up" if change >= 0 else "down"
    return f"Price moved {direction} {abs(change):.2f}%"


# --- Create the agent ---
agent = create_deep_agent(
    model=llm,
    tools=[get_stock_price, calculate_percentage_change],
    system_prompt="You are a financial assistant. Use the available tools to answer questions about stock prices.",
)

# --- Run it ---
result = agent.invoke({"messages": "What is the current price of AAPL and MSFT? Which one is higher?"})
print(result["messages"][-1].content)

---
## Example 2 — Planning with `write_todos`

For complex tasks the agent automatically calls `write_todos` to decompose the objective into steps before starting. You can inspect the plan by watching the intermediate messages.

In [ ]:
import json
from langchain_core.tools import tool
from deepagents import create_deep_agent

@tool
def search_web(query: str) -> str:
    """Search the web and return a short summary of results."""
    # Simulated search — replace with Tavily or SerpAPI in production
    results = {
        "python web frameworks": "Top Python web frameworks: Django (full-stack), FastAPI (async APIs), Flask (micro), Starlette (lightweight async), Tornado (non-blocking).",
        "django performance": "Django handles ~5k–20k req/s on typical hardware. Use caching, DB indexing, and async views for scale.",
        "fastapi performance": "FastAPI handles ~30k–100k req/s. Built on Starlette + Pydantic. Best for async, high-throughput APIs.",
        "flask performance": "Flask handles ~3k–8k req/s in sync mode. Simple and flexible, best for small/medium projects.",
    }
    for key, val in results.items():
        if key in query.lower():
            return val
    return f"Search results for '{query}': No specific data available in simulation."


# --- Agent with planning enabled (default) ---
planning_agent = create_deep_agent(
    model=llm,
    tools=[search_web],
    system_prompt="""You are a research analyst. 
    For any research task, ALWAYS start by calling write_todos to plan your steps.
    Then execute each step methodically.""",
)

# --- Run a multi-step research task ---
result = planning_agent.invoke({
    "messages": "Compare Django, FastAPI, and Flask. Cover: use cases, performance, and which to pick for a new REST API project."
})

# Print all messages to see the planning steps
for msg in result["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    if hasattr(msg, "content") and msg.content:
        print(f"[{role}]\n{msg.content}\n{'-'*60}")

---
## Example 3 — Filesystem: Read, Write, and Edit Files

DeepAgents ships built-in filesystem tools (`read_file`, `write_file`, `edit_file`, `list_dir`, `find_files`). Here the agent generates a Python module, saves it, then edits it on request.

In [ ]:
import os
from deepagents import create_deep_agent
from deepagents.filesystem import LocalFilesystemBackend

# Create a workspace directory for the agent to use
workspace = "/tmp/deepagent_workspace"
os.makedirs(workspace, exist_ok=True)

# --- Agent with local filesystem backend ---
fs_agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="""You are a Python developer assistant. 
    Use the filesystem tools to create, read, and edit Python files.
    Always write clean, type-annotated Python code.""",
    filesystem_backend=LocalFilesystemBackend(root=workspace),
)

# --- Step 1: Generate a utility module ---
result = fs_agent.invoke({
    "messages": """Create a file called 'stats_utils.py' with these functions:
    1. mean(values: list[float]) -> float
    2. median(values: list[float]) -> float  
    3. std_dev(values: list[float]) -> float
    Each function should handle empty lists gracefully."""
})
print("Step 1 done:", result["messages"][-1].content)
print()

# --- Step 2: Read it back ---
result2 = fs_agent.invoke({
    "messages": "Read the file 'stats_utils.py' and show me its contents."
})
print("Step 2 — File contents:\n")
print(result2["messages"][-1].content)

In [ ]:
# --- Step 3: Edit the file ---
result3 = fs_agent.invoke({
    "messages": "Add a 'variance(values: list[float]) -> float' function to 'stats_utils.py'."
})
print("Step 3 — After edit:\n")
print(result3["messages"][-1].content)

# Verify on disk
with open(f"{workspace}/stats_utils.py") as f:
    print("\n--- Actual file on disk ---")
    print(f.read())

---
## Example 4 — Subagent Delegation

An orchestrator agent delegates independent subtasks to specialist child agents. Each child runs in an **isolated context window** and returns only its final result — keeping the orchestrator's context clean.

In [ ]:
from langchain_core.tools import tool
from deepagents import create_deep_agent

# --- Shared simulated data tools ---
@tool
def get_company_financials(company: str) -> str:
    """Retrieve revenue, profit, and growth data for a company."""
    data = {
        "openai": "Revenue: $3.4B (2024), Growth: 200% YoY, Profitable: No (heavy compute costs)",
        "anthropic": "Revenue: $1.1B (2024), Growth: 300% YoY, Profitable: No (research-heavy)",
        "google": "Revenue: $307B (2024), Growth: 8% YoY, Profitable: Yes (operating margin 31%)",
    }
    return data.get(company.lower(), f"No financial data for {company}.")

@tool
def get_company_products(company: str) -> str:
    """List the key products and services of a company."""
    data = {
        "openai": "ChatGPT, GPT-4o, DALL-E 3, Sora, Whisper, Codex, OpenAI API",
        "anthropic": "Claude 4 family (Haiku/Sonnet/Opus), Claude API, Constitutional AI research",
        "google": "Search, Ads, YouTube, Android, GCP, Gemini 2.5, Workspace, Maps",
    }
    return data.get(company.lower(), f"No product data for {company}.")

@tool
def get_market_sentiment(company: str) -> str:
    """Get analyst sentiment and recent news about a company."""
    data = {
        "openai": "Bullish: dominant mindshare, broad enterprise adoption. Risk: compute costs, competition.",
        "anthropic": "Bullish: safety-focused differentiation, strong enterprise deals. Risk: smaller scale.",
        "google": "Neutral-Bullish: strong fundamentals, AI integration efforts. Risk: search disruption.",
    }
    return data.get(company.lower(), f"No sentiment data for {company}.")


# --- Specialist subagents ---
financial_analyst = create_deep_agent(
    model=llm,
    tools=[get_company_financials],
    system_prompt="You are a financial analyst. Analyze company financials and provide concise summaries with key metrics.",
)

product_analyst = create_deep_agent(
    model=llm,
    tools=[get_company_products, get_market_sentiment],
    system_prompt="You are a product and market analyst. Evaluate product portfolios and market positioning.",
)

# --- Orchestrator ---
orchestrator = create_deep_agent(
    model=llm,
    tools=[],
    subagents={
        "financial_analyst": financial_analyst,
        "product_analyst": product_analyst,
    },
    system_prompt="""You are a senior investment research director.
    Delegate analysis tasks to your specialist subagents, then synthesize
    their findings into a structured investment brief.""",
)

# --- Run the orchestrated analysis ---
result = orchestrator.invoke({
    "messages": """Produce a comparative analysis of OpenAI, Anthropic, and Google 
    from an investment perspective. Cover financials, products, and market sentiment 
    for each. Conclude with a recommendation."""
})

print(result["messages"][-1].content)

---
## Example 5 — Streaming Output

DeepAgents inherits LangGraph's streaming. Use `astream_events` to surface tokens, tool calls, and tool results as they happen — essential for interactive UIs.

In [ ]:
import asyncio
from langchain_core.tools import tool
from deepagents import create_deep_agent

@tool
def get_country_facts(country: str) -> str:
    """Return key facts about a country: capital, population, GDP."""
    facts = {
        "france": "Capital: Paris | Population: 68M | GDP: $3.0T | Language: French",
        "japan": "Capital: Tokyo | Population: 125M | GDP: $4.2T | Language: Japanese",
        "brazil": "Capital: Brasília | Population: 215M | GDP: $2.1T | Language: Portuguese",
    }
    return facts.get(country.lower(), f"No data for {country}.")


stream_agent = create_deep_agent(
    model=llm,
    tools=[get_country_facts],
    system_prompt="You are a geography assistant. Provide detailed, engaging answers about countries.",
)


async def stream_response(query: str):
    """Stream the agent response, printing tokens and tool events live."""
    print(f"Query: {query}\n{'='*60}")

    async for event in stream_agent.astream_events(
        {"messages": query},
        version="v2",
    ):
        event_type = event["event"]

        # Live token streaming
        if event_type == "on_chat_model_stream":
            token = event["data"]["chunk"].content
            if token:
                print(token, end="", flush=True)

        # Tool call notification
        elif event_type == "on_tool_start":
            tool_name = event["name"]
            tool_input = event["data"].get("input", {})
            print(f"\n\n[Tool called: {tool_name}({tool_input})]")

        # Tool result
        elif event_type == "on_tool_end":
            print(f"[Tool result: {event['data'].get('output', '')}]\n")

    print("\n" + "="*60)


# Run the async streamer
await stream_response("Compare France, Japan, and Brazil. Which has the highest GDP per capita?")

---
## Example 6 — Multi-Step Research Pipeline (Full Deep Agent Pattern)

This example shows the canonical DeepAgents pattern for real work:
- **Orchestrator** uses `write_todos` to plan
- Delegates independent research topics to **specialist subagents**
- Synthesizes all findings into a **structured report saved to disk**

In [ ]:
import os
from langchain_core.tools import tool
from deepagents import create_deep_agent
from deepagents.filesystem import LocalFilesystemBackend

# Workspace for the report output
report_dir = "/tmp/research_output"
os.makedirs(report_dir, exist_ok=True)

# ---------------------------------------------------------------------------
# Simulated research tools (replace with Tavily / SerpAPI in production)
# ---------------------------------------------------------------------------

@tool
def search_academic(topic: str) -> str:
    """Search academic papers and research on a topic."""
    db = {
        "transformer architecture": (
            "'Attention is All You Need' (Vaswani et al., 2017) introduced the Transformer. "
            "Key innovations: multi-head self-attention, positional encoding, feed-forward layers. "
            "~100k citations. Foundation of all modern LLMs."
        ),
        "llm scaling laws": (
            "Kaplan et al. (2020) showed compute-optimal scaling: loss decreases as power law "
            "with model size and data. Chinchilla (Hoffmann et al., 2022) refined this — "
            "models were undertrained; optimal: equal scaling of params and tokens."
        ),
        "rlhf alignment": (
            "RLHF (Christiano et al., 2017) trains a reward model from human preferences, "
            "then fine-tunes the LLM via PPO. Used in InstructGPT, ChatGPT, Claude. "
            "DPO (Rafailov et al., 2023) simplifies this without an explicit reward model."
        ),
    }
    for key, val in db.items():
        if key in topic.lower():
            return val
    return f"Academic search for '{topic}': Extensive literature exists; key papers available via arXiv and Semantic Scholar."


@tool
def search_industry(topic: str) -> str:
    """Search industry reports and market data on a topic."""
    db = {
        "llm market size": (
            "Global LLM market: $6.4B in 2024, projected $140B by 2030 (CAGR 47%). "
            "Key players: OpenAI, Google, Anthropic, Meta, Mistral. "
            "Enterprise adoption growing 3x YoY."
        ),
        "ai agent adoption": (
            "AI agents market: $3.8B in 2024, projected $50B by 2030. "
            "Top use cases: coding (GitHub Copilot, Cursor), customer service, research. "
            "60% of Fortune 500 piloting agentic workflows."
        ),
        "llm inference costs": (
            "Inference costs dropped 90% from 2023–2025 (Wright's Law). "
            "GPT-4 equivalent: $30/M tokens (2023) → $0.15/M tokens (2026). "
            "Driving commoditization and new application economics."
        ),
    }
    for key, val in db.items():
        if key in topic.lower():
            return val
    return f"Industry search for '{topic}': Reports from Gartner, IDC, and a16z provide coverage."


# ---------------------------------------------------------------------------
# Specialist subagents
# ---------------------------------------------------------------------------

academic_researcher = create_deep_agent(
    model=llm,
    tools=[search_academic],
    system_prompt="""You are an academic researcher specialising in AI/ML.
    When given a research question, use search_academic to find relevant papers 
    and provide a concise, well-cited summary (3–5 sentences).""",
)

market_researcher = create_deep_agent(
    model=llm,
    tools=[search_industry],
    system_prompt="""You are a market research analyst.
    When given a topic, use search_industry to retrieve market data 
    and summarise key figures, trends, and implications (3–5 sentences).""",
)

# ---------------------------------------------------------------------------
# Orchestrator with filesystem for saving the report
# ---------------------------------------------------------------------------

orchestrator = create_deep_agent(
    model=llm,
    tools=[],
    subagents={
        "academic_researcher": academic_researcher,
        "market_researcher": market_researcher,
    },
    system_prompt="""You are a senior research director producing an industry report.

    Workflow:
    1. Call write_todos to plan your research steps.
    2. Delegate each research question to the appropriate subagent:
       - academic questions → academic_researcher
       - market/industry questions → market_researcher
    3. Synthesise all findings into a structured Markdown report.
    4. Save the final report to 'report.md' using write_file.
    5. Return a brief summary of what you found.""",
    filesystem_backend=LocalFilesystemBackend(root=report_dir),
)

# ---------------------------------------------------------------------------
# Run the full pipeline
# ---------------------------------------------------------------------------

print("Starting deep research pipeline...\n")

result = orchestrator.invoke({
    "messages": """Write a comprehensive report titled 'The State of Large Language Models in 2026'. 
    Cover:
    - Technical foundations (transformer architecture, scaling laws)
    - Alignment techniques (RLHF)
    - Market size and growth projections
    - AI agent adoption trends
    - Inference cost trends
    Save the report to report.md."""
})

print("\n" + "="*60)
print("ORCHESTRATOR SUMMARY:")
print(result["messages"][-1].content)

In [ ]:
# Read back the saved report
report_path = f"{report_dir}/report.md"
if os.path.exists(report_path):
    with open(report_path) as f:
        print(f.read())
else:
    print("Report file not found — check the agent's filesystem write step.")

---
## Summary

| Example | DeepAgents Feature | Key API |
|---|---|---|
| 1 — Basic Agent | Custom tools | `create_deep_agent(model, tools, system_prompt)` |
| 2 — Planning | Automatic task decomposition | `write_todos` (built-in, automatic) |
| 3 — Filesystem | Read / write / edit files | `LocalFilesystemBackend`, built-in file tools |
| 4 — Subagents | Isolated specialist delegation | `subagents={name: agent}`, `task` tool |
| 5 — Streaming | Live token & event stream | `agent.astream_events(…, version="v2")` |
| 6 — Pipeline | Full orchestration pattern | All of the above combined |

### Next steps

- Add **persistent memory** with `AGENTS.md` to make preferences survive across sessions.
- Add **SKILL.md** files to load specialised instructions on demand.
- Switch to **async subagents** (`AsyncSubagentConfig`) for tasks that take minutes to hours.
- Enable **human-in-the-loop** with `ApprovalConfig` to review sensitive tool calls.
- Deploy via **LangGraph Cloud** or **Managed Deep Agents** for production.